In [133]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [134]:
PJME_data = pd.read_csv(
    "PJME_phase2_preprocessed.csv",
    parse_dates=["Datetime"]
)

In [135]:
PJME_data = PJME_data.set_index("Datetime")

# Sort chronologically
PJME_data = PJME_data.sort_index()

print(PJME_data.shape)
print(PJME_data.head())

(145224, 15)
                     PJME_MW  Hour  Day  Week  Month  DayOfWeek  IsWeekend  \
Datetime                                                                     
2002-01-08 01:00:00  29445.0     1    8     2      1          1          0   
2002-01-08 02:00:00  28670.0     2    8     2      1          1          0   
2002-01-08 03:00:00  28375.0     3    8     2      1          1          0   
2002-01-08 04:00:00  28542.0     4    8     2      1          1          0   
2002-01-08 05:00:00  29261.0     5    8     2      1          1          0   

                       Lag_1   Lag_24   Lag_48  Lag_168  Rolling_Mean_24  \
Datetime                                                                   
2002-01-08 01:00:00  31187.0  26862.0  27100.0  30393.0     33452.583333   
2002-01-08 02:00:00  29445.0  25976.0  26097.0  29265.0     33560.208333   
2002-01-08 03:00:00  28670.0  25641.0  25793.0  28357.0     33672.458333   
2002-01-08 04:00:00  28375.0  25666.0  25657.0  27899.0     

In [136]:
# Make sure data is chronological
PJME_data = PJME_data.sort_index()

n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = PJME_data.iloc[:train_end].copy()
validation = PJME_data.iloc[train_end:val_end].copy()
test = PJME_data.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

print("\nTrain period:")
print(train.index.min(), "to", train.index.max())

print("\nValidation period:")
print(validation.index.min(), "to", validation.index.max())

print("\nTest period:")
print(test.index.min(), "to", test.index.max())

Train: (101656, 15)
Validation: (21784, 15)
Test: (21784, 15)

Train period:
2002-01-08 01:00:00 to 2013-08-13 16:00:00

Validation period:
2013-08-13 17:00:00 to 2016-02-07 08:00:00

Test period:
2016-02-07 09:00:00 to 2018-08-03 00:00:00


In [139]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

target = "PJME_MW"

X_train = train[features].copy()
X_val = validation[features].copy()
X_test = test[features].copy()

y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (101656, 15)
X_val: (21784, 15)
X_test: (21784, 15)


In [141]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = target_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = target_scaler.transform(
    y_test.values.reshape(-1, 1)
)

print("Scaling completed.")

Scaling completed.


In [142]:
def create_sequences(X, y, input_steps=24, output_steps=24):

    X_seq = []
    y_seq = []

    for i in range(input_steps, len(X) - output_steps + 1):

        X_seq.append(
            X[i-input_steps:i]
        )

        y_seq.append(
            y[i:i+output_steps].flatten()
        )

    return np.array(X_seq), np.array(y_seq)


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=24,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=24,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=24,
    output_steps=24
)


print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101609, 24, 15)
y_train: (101609, 24)
X_val: (21737, 24, 15)
y_val: (21737, 24)
X_test: (21737, 24, 15)
y_test: (21737, 24)


**Early Stopping**

In [143]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [144]:
rnn_es_model = Sequential([
    SimpleRNN(
        64,
        input_shape=(24, 15),
        return_sequences=False
    ),

    Dense(64, activation="relu"),
    Dense(24)
])

rnn_es_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_es_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_7 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [145]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [146]:
history_es = rnn_es_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.1299 - mae: 0.2601 - val_loss: 0.0978 - val_mae: 0.2276
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0801 - mae: 0.2046 - val_loss: 0.0851 - val_mae: 0.2099
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0727 - mae: 0.1928 - val_loss: 0.0833 - val_mae: 0.2088
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0700 - mae: 0.1881 - val_loss: 0.0889 - val_mae: 0.2150
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0679 - mae: 0.1844 - val_loss: 0.0818 - val_mae: 0.2041
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0663 - mae: 0.1816 - val_loss: 0.0871 - val_mae: 0.2156
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0652 - mae: 0.1797 - val_loss: 0.0786 - val_mae: 0.1990
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0641 - mae: 0.1781 - val_loss: 0.0751 - val_mae: 0.1942
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━

In [147]:
rnn_es_pred_scaled = rnn_es_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [148]:
rnn_es_pred = target_scaler.inverse_transform(
    rnn_es_pred_scaled.reshape(-1, 1)
).reshape(rnn_es_pred_scaled.shape)

rnn_es_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [149]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(
    rnn_es_actual.flatten(),
    rnn_es_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        rnn_es_actual.flatten(),
        rnn_es_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (rnn_es_actual.flatten() - rnn_es_pred.flatten())
        / rnn_es_actual.flatten()
    )
) * 100

r2 = r2_score(
    rnn_es_actual.flatten(),
    rnn_es_pred.flatten()
)

bias = np.mean(
    rnn_es_pred.flatten() - rnn_es_actual.flatten()
)

print("==== RNN + EARLY STOPPING ====")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

==== RNN + EARLY STOPPING ====
MAE : 1390.3678448612964
RMSE: 1949.1159618860881
MAPE: 4.410774413858794
R²  : 0.9084643650051681
Bias: 120.65475509308246


In [150]:
print("Epochs completed:", len(history_es.history["loss"]))

Epochs completed: 15


**Dropout**

In [151]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout

rnn_dropout_model = Sequential([
    SimpleRNN(
        64,
        input_shape=(24, 15),
        return_sequences=False
    ),

    Dropout(0.2),

    Dense(64, activation="relu"),

    Dropout(0.2),

    Dense(24)
])

rnn_dropout_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_dropout_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_8 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [152]:
history_dropout = rnn_dropout_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - loss: 0.2052 - mae: 0.3383 - val_loss: 0.1052 - val_mae: 0.2463
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1331 - mae: 0.2764 - val_loss: 0.0927 - val_mae: 0.2263
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1263 - mae: 0.2681 - val_loss: 0.0893 - val_mae: 0.2199
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.1222 - mae: 0.2635 - val_loss: 0.0887 - val_mae: 0.2203
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1197 - mae: 0.2608 - val_loss: 0.0870 - val_mae: 0.2175
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1181 - mae: 0.2590 - val_loss: 0.0870 - val_mae: 0.2148
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.1165 - mae: 0.2568 - val_loss: 0.0829 - val_mae: 0.2080
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1156 - mae: 0.2559 - val_loss: 0.0883 - val_mae: 0.2195
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━━

In [153]:
dropout_pred_scaled = rnn_dropout_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [154]:
dropout_pred = target_scaler.inverse_transform(
    dropout_pred_scaled.reshape(-1, 1)
).reshape(dropout_pred_scaled.shape)

dropout_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [155]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(
    dropout_actual.flatten(),
    dropout_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        dropout_actual.flatten(),
        dropout_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (dropout_actual.flatten() - dropout_pred.flatten())
        / dropout_actual.flatten()
    )
) * 100

r2 = r2_score(
    dropout_actual.flatten(),
    dropout_pred.flatten()
)

bias = np.mean(
    dropout_pred.flatten() -
    dropout_actual.flatten()
)

print("=== RNN + DROPOUT ===")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

=== RNN + DROPOUT ===
MAE : 1471.529424933611
RMSE: 2009.527278024123
MAPE: 4.691583628341732
R²  : 0.9027022820804059
Bias: 117.89846316413617


**Batch Normalization**

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, BatchNormalization

rnn_bn_model = Sequential([
    SimpleRNN(
        64,
        input_shape=(24, 15)
    ),

    BatchNormalization(),

    Dense(64, activation="relu"),

    BatchNormalization(),

    Dense(24)
])

rnn_bn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_bn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_2 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,352 (44.34 KB)

 Trainable params: 11,096 (43.34 KB)

 Non-trainable params: 256 (1.00 KB)

In [24]:
history_bn = rnn_bn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - loss: 0.1704 - mae: 0.3025 - val_loss: 0.0996 - val_mae: 0.2342
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - loss: 0.1007 - mae: 0.2374 - val_loss: 0.0931 - val_mae: 0.2261
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0935 - mae: 0.2277 - val_loss: 0.0871 - val_mae: 0.2110
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0894 - mae: 0.2221 - val_loss: 0.0867 - val_mae: 0.2100
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0863 - mae: 0.2178 - val_loss: 0.0784 - val_mae: 0.2032
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0853 - mae: 0.2165 - val_loss: 0.0768 - val_mae: 0.1994
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0840 - mae: 0.2148 - val_loss: 0.0771 - val_mae: 0.1975
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0821 - mae: 0.2124 - val_loss: 0.0783 - val_mae: 0.2009
Epoch 9/15
1588/1588 ━━━━━━━━━━━━

In [26]:
bn_pred_scaled = rnn_bn_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [27]:
rnn_bn_pred = target_scaler.inverse_transform(
    bn_pred_scaled.reshape(-1, 1)
).reshape(rnn_es_pred_scaled.shape)

rnn_bn_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(
    rnn_bn_actual.flatten(),
    rnn_bn_pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        rnn_bn_actual.flatten(),
        rnn_bn_pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (rnn_bn_actual.flatten() - rnn_bn_pred.flatten())
        / rnn_bn_actual.flatten()
    )
) * 100

r2 = r2_score(
    rnn_bn_actual.flatten(),
    rnn_bn_pred.flatten()
)

bias = np.mean(
    rnn_bn_pred.flatten() - rnn_bn_actual.flatten()
)

print("==== RNN + Batch ====")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

==== RNN + Batch ====
MAE : 1388.6084414431793
RMSE: 1960.869031599301
MAPE: 4.408847433340167
R²  : 0.9073571263344717
Bias: 216.50879045895846


**ADAM**

In [29]:
from tensorflow.keras.optimizers import Adam

rnn_adam = Sequential([
    SimpleRNN(64, input_shape=(24, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_adam.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [30]:
history_adam = rnn_adam.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 9ms/step - loss: 0.1206 - mae: 0.2494 - val_loss: 0.0956 - val_mae: 0.2262
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - loss: 0.0769 - mae: 0.1994 - val_loss: 0.0869 - val_mae: 0.2122
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0712 - mae: 0.1900 - val_loss: 0.0813 - val_mae: 0.2040
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0682 - mae: 0.1852 - val_loss: 0.0803 - val_mae: 0.2024
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0664 - mae: 0.1820 - val_loss: 0.0775 - val_mae: 0.1992
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0651 - mae: 0.1800 - val_loss: 0.0784 - val_mae: 0.1986
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0641 - mae: 0.1785 - val_loss: 0.0756 - val_mae: 0.1974
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0628 - mae: 0.1762 - val_loss: 0.0778 - val_mae: 0.2001
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━

In [31]:
adam_pred_scaled = rnn_adam.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [32]:
adam_pred = target_scaler.inverse_transform(
    adam_pred_scaled.reshape(-1, 1)
).reshape(adam_pred_scaled.shape)

adam_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [33]:
adam_mae = mean_absolute_error(
    adam_actual.flatten(),
    adam_pred.flatten()
)

adam_rmse = np.sqrt(
    mean_squared_error(
        adam_actual.flatten(),
        adam_pred.flatten()
    )
)

adam_mape = np.mean(
    np.abs(
        (adam_actual.flatten() - adam_pred.flatten())
        / adam_actual.flatten()
    )
) * 100

adam_r2 = r2_score(
    adam_actual.flatten(),
    adam_pred.flatten()
)

adam_bias = np.mean(
    adam_pred.flatten() -
    adam_actual.flatten()
)

print("========== RNN + ADAM ==========")
print("MAE :", adam_mae)
print("RMSE:", adam_rmse)
print("MAPE:", adam_mape)
print("R²  :", adam_r2)
print("Bias:", adam_bias)

========== RNN + ADAM ==========
MAE : 1412.8646858513553
RMSE: 1960.2076349436081
MAPE: 4.477067885744956
R²  : 0.907419612254838
Bias: 138.76057845742451


**RMSPROP**

In [34]:
from tensorflow.keras.optimizers import RMSprop

rnn_rmsprop = Sequential([
    SimpleRNN(
        64,
        input_shape=(24, 15),
        return_sequences=False
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

rnn_rmsprop.compile(
    optimizer=RMSprop(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

rnn_rmsprop.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_4 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
history_rmsprop = rnn_rmsprop.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - loss: 0.1237 - mae: 0.2557 - val_loss: 0.0980 - val_mae: 0.2311
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 0.0801 - mae: 0.2039 - val_loss: 0.0896 - val_mae: 0.2152
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0733 - mae: 0.1929 - val_loss: 0.0840 - val_mae: 0.2079
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0697 - mae: 0.1870 - val_loss: 0.0799 - val_mae: 0.2019
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0676 - mae: 0.1835 - val_loss: 0.0851 - val_mae: 0.2113
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 0.0658 - mae: 0.1806 - val_loss: 0.0781 - val_mae: 0.2027
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0647 - mae: 0.1787 - val_loss: 0.0777 - val_mae: 0.1992
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0637 - mae: 0.1770 - val_loss: 0.0778 - val_mae: 0.1976
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━

In [36]:
rmsprop_pred_scaled = rnn_rmsprop.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rmsprop_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Prediction shape: (21737, 24)


In [37]:
rmsprop_pred = target_scaler.inverse_transform(
    rmsprop_pred_scaled.reshape(-1, 1)
).reshape(rmsprop_pred_scaled.shape)

rmsprop_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [38]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rmsprop_mae = mean_absolute_error(
    rmsprop_actual.flatten(),
    rmsprop_pred.flatten()
)

rmsprop_rmse = np.sqrt(
    mean_squared_error(
        rmsprop_actual.flatten(),
        rmsprop_pred.flatten()
    )
)

rmsprop_mape = np.mean(
    np.abs(
        (rmsprop_actual.flatten() - rmsprop_pred.flatten())
        / rmsprop_actual.flatten()
    )
) * 100

rmsprop_r2 = r2_score(
    rmsprop_actual.flatten(),
    rmsprop_pred.flatten()
)

rmsprop_bias = np.mean(
    rmsprop_pred.flatten() -
    rmsprop_actual.flatten()
)

print("==== RNN + RMSPROP ====")
print("MAE :", rmsprop_mae)
print("RMSE:", rmsprop_rmse)
print("MAPE:", rmsprop_mape)
print("R²  :", rmsprop_r2)
print("Bias:", rmsprop_bias)

==== RNN + RMSPROP ====
MAE : 1387.1081347502004
RMSE: 1931.1706843661027
MAPE: 4.41234224260603
R²  : 0.9101421211213101
Bias: 273.0465344812776


**SGD**

In [39]:
from tensorflow.keras.optimizers import SGD

rnn_sgd = Sequential([
    SimpleRNN(
        64,
        input_shape=(24, 21),
        return_sequences=False
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

rnn_sgd.compile(
    optimizer=SGD(
        learning_rate=0.001,
        momentum=0.9
    ),
    loss="mse",
    metrics=["mae"]
)

rnn_sgd.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_5 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,840 (42.34 KB)

 Trainable params: 10,840 (42.34 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
history_sgd = rnn_sgd.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.4068 - mae: 0.4850 - val_loss: 0.2266 - val_mae: 0.3737
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1799 - mae: 0.3283 - val_loss: 0.1682 - val_mae: 0.3157
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.1419 - mae: 0.2874 - val_loss: 0.1463 - val_mae: 0.2925
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1261 - mae: 0.2688 - val_loss: 0.1333 - val_mae: 0.2767
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.1171 - mae: 0.2577 - val_loss: 0.1267 - val_mae: 0.2681
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.1110 - mae: 0.2499 - val_loss: 0.1209 - val_mae: 0.2605
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.1063 - mae: 0.2437 - val_loss: 0.1168 - val_mae: 0.2550
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.1025 - mae: 0.2386 - val_loss: 0.1132 - val_mae: 0.2510
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━

In [41]:
sgd_pred_scaled = rnn_sgd.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", sgd_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21737, 24)


In [42]:
sgd_pred = target_scaler.inverse_transform(
    sgd_pred_scaled.reshape(-1, 1)
).reshape(sgd_pred_scaled.shape)

sgd_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [43]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

sgd_mae = mean_absolute_error(
    sgd_actual.flatten(),
    sgd_pred.flatten()
)

sgd_rmse = np.sqrt(
    mean_squared_error(
        sgd_actual.flatten(),
        sgd_pred.flatten()
    )
)

sgd_mape = np.mean(
    np.abs(
        (sgd_actual.flatten() - sgd_pred.flatten())
        / sgd_actual.flatten()
    )
) * 100

sgd_r2 = r2_score(
    sgd_actual.flatten(),
    sgd_pred.flatten()
)

sgd_bias = np.mean(
    sgd_pred.flatten() -
    sgd_actual.flatten()
)
print("=== RNN + SGD ===")
print("MAE :", sgd_mae)
print("RMSE:", sgd_rmse)
print("MAPE:", sgd_mape)
print("R²  :", sgd_r2)
print("Bias:", sgd_bias)

=== RNN + SGD ===
MAE : 1611.0813879169339
RMSE: 2180.9567945231242
MAPE: 5.1341236813930715
R²  : 0.8853935746511137
Bias: 159.78651111047816


**Learning Rate Scheduling**

In [44]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import RMSprop

rnn_lr = Sequential([
    SimpleRNN(
        64,
        input_shape=(24, 15)
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

rnn_lr.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [45]:
history_lr = rnn_lr.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[lr_scheduler],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - loss: 0.1245 - mae: 0.2558 - val_loss: 0.0960 - val_mae: 0.2236 - learning_rate: 0.0010
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0811 - mae: 0.2053 - val_loss: 0.0941 - val_mae: 0.2212 - learning_rate: 0.0010
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0747 - mae: 0.1951 - val_loss: 0.0824 - val_mae: 0.2055 - learning_rate: 0.0010
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0712 - mae: 0.1892 - val_loss: 0.0862 - val_mae: 0.2132 - learning_rate: 0.0010
Epoch 5/15
1587/1588 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0692 - mae: 0.1860
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0690 - mae: 0.1855 - val_loss: 0.0854 - val_mae: 0.2106 - learning_rate: 0.0010
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0621 - mae: 0.1736 - val_loss: 0.0755 - val_mae: 0.1943 - learnin

In [46]:
lr_pred_scaled = rnn_lr.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lr_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21737, 24)


In [47]:
lr_pred = target_scaler.inverse_transform(
    lr_pred_scaled.reshape(-1, 1)
).reshape(lr_pred_scaled.shape)

lr_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [48]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lr_mae = mean_absolute_error(
    lr_actual.flatten(),
    lr_pred.flatten()
)

lr_rmse = np.sqrt(
    mean_squared_error(
        lr_actual.flatten(),
        lr_pred.flatten()
    )
)

lr_mape = np.mean(
    np.abs(
        (lr_actual.flatten() - lr_pred.flatten())
        / lr_actual.flatten()
    )
) * 100

lr_r2 = r2_score(
    lr_actual.flatten(),
    lr_pred.flatten()
)

lr_bias = np.mean(
    lr_pred.flatten() -
    lr_actual.flatten()
)

print("==== RNN + RMSPROP + LearningRate ====")
print("MAE :", lr_mae)
print("RMSE:", lr_rmse)
print("MAPE:", lr_mape)
print("R²  :", lr_r2)
print("Bias:", lr_bias)

==== RNN + RMSPROP + LearningRate ====
MAE : 1380.455857086755
RMSE: 1947.0097018210615
MAPE: 4.373047915495958
R²  : 0.9086620891892203
Bias: 183.9371604022663


**layers, neurons**

In [49]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_deep = Sequential([

    SimpleRNN(
        64,
        return_sequences=True,
        input_shape=(24, 15)
    ),

    SimpleRNN(
        32,
        return_sequences=False
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

rnn_deep.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_deep.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_7 (SimpleRNN)        │ (None, 24, 64)         │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_8 (SimpleRNN)        │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,896 (46.47 KB)

 Trainable params: 11,896 (46.47 KB)

 Non-trainable params: 0 (0.00 B)

In [50]:
history_deep = rnn_deep.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 24s 9ms/step - loss: 0.1299 - mae: 0.2585 - val_loss: 0.0977 - val_mae: 0.2267
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0781 - mae: 0.2018 - val_loss: 0.0873 - val_mae: 0.2136
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - loss: 0.0725 - mae: 0.1928 - val_loss: 0.0918 - val_mae: 0.2203
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0695 - mae: 0.1879 - val_loss: 0.0807 - val_mae: 0.2046
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0679 - mae: 0.1853 - val_loss: 0.0815 - val_mae: 0.2074
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0661 - mae: 0.1826 - val_loss: 0.0799 - val_mae: 0.2036
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0653 - mae: 0.1816 - val_loss: 0.0805 - val_mae: 0.2032
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0639 - mae: 0.1793 - val_loss: 0.0764 - val_mae: 0.2003
Epoch 9/15
1588/1588 ━━━━━━━━━━

In [51]:
deep_pred_scaled = rnn_deep.predict(
    X_test_seq,
    batch_size=64
)
print("Prediction shape:", deep_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Prediction shape: (21737, 24)


In [52]:
deep_pred = target_scaler.inverse_transform(
    deep_pred_scaled.reshape(-1, 1)
).reshape(deep_pred_scaled.shape)

deep_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [53]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

deep_mae = mean_absolute_error(
    deep_actual.flatten(),
    deep_pred.flatten()
)

deep_rmse = np.sqrt(
    mean_squared_error(
        deep_actual.flatten(),
        deep_pred.flatten()
    )
)

deep_mape = np.mean(
    np.abs(
        (deep_actual.flatten() - deep_pred.flatten())
        / deep_actual.flatten()
    )
) * 100

deep_r2 = r2_score(
    deep_actual.flatten(),
    deep_pred.flatten()
)

deep_bias = np.mean(
    deep_pred.flatten() -
    deep_actual.flatten()
)

print("========== DEEP RNN ==========")
print("MAE :", deep_mae)
print("RMSE:", deep_rmse)
print("MAPE:", deep_mape)
print("R²  :", deep_r2)
print("Bias:", deep_bias)

========== DEEP RNN ==========
MAE : 1499.4187534323676
RMSE: 2093.1793458589927
MAPE: 4.784466855917797
R²  : 0.8944331108299858
Bias: 383.87631982172536


In [54]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_128 = Sequential([

    SimpleRNN(
        128,
        input_shape=(24, 15)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

rnn_128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_128.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_9 (SimpleRNN)        │ (None, 128)            │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,248 (110.34 KB)

 Trainable params: 28,248 (110.34 KB)

 Non-trainable params: 0 (0.00 B)

In [55]:
history_128 = rnn_128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 6ms/step - loss: 0.1094 - mae: 0.2363 - val_loss: 0.0876 - val_mae: 0.2153
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0726 - mae: 0.1925 - val_loss: 0.0862 - val_mae: 0.2106
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0682 - mae: 0.1855 - val_loss: 0.0811 - val_mae: 0.2038
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0651 - mae: 0.1801 - val_loss: 0.0776 - val_mae: 0.1992
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0640 - mae: 0.1783 - val_loss: 0.0783 - val_mae: 0.1995
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0650 - mae: 0.1799 - val_loss: 0.0762 - val_mae: 0.1955
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0617 - mae: 0.1745 - val_loss: 0.0751 - val_mae: 0.1953
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0604 - mae: 0.1722 - val_loss: 0.0732 - val_mae: 0.1943
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━

In [56]:
rnn128_pred_scaled = rnn_128.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rnn128_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21737, 24)


In [57]:
rnn128_pred = target_scaler.inverse_transform(
    rnn128_pred_scaled.reshape(-1, 1)
).reshape(rnn128_pred_scaled.shape)

rnn128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [58]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rnn128_mae = mean_absolute_error(
    rnn128_actual.flatten(),
    rnn128_pred.flatten()
)

rnn128_rmse = np.sqrt(
    mean_squared_error(
        rnn128_actual.flatten(),
        rnn128_pred.flatten()
    )
)

rnn128_mape = np.mean(
    np.abs(
        (rnn128_actual.flatten() - rnn128_pred.flatten())
        / rnn128_actual.flatten()
    )
) * 100

rnn128_r2 = r2_score(
    rnn128_actual.flatten(),
    rnn128_pred.flatten()
)

rnn128_bias = np.mean(
    rnn128_pred.flatten() -
    rnn128_actual.flatten()
)

print("=== RNN 128 NEURONS ===")
print("MAE :", rnn128_mae)
print("RMSE:", rnn128_rmse)
print("MAPE:", rnn128_mape)
print("R²  :", rnn128_r2)
print("Bias:", rnn128_bias)

=== RNN 128 NEURONS ===
MAE : 1458.0408453731266
RMSE: 2046.0232695734032
MAPE: 4.657677636162435
R²  : 0.8991360481426458
Bias: 522.2727100125278


**32 neurons**

In [59]:
rnn_32 = Sequential([
    SimpleRNN(
        32,
        input_shape=(24, 15)
    ),

    Dense(64, activation="relu"),

    Dense(24)
])

rnn_32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_32.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_10 (SimpleRNN)       │ (None, 32)             │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,208 (20.34 KB)

 Trainable params: 5,208 (20.34 KB)

 Non-trainable params: 0 (0.00 B)

In [60]:
history_32 = rnn_32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - loss: 0.1459 - mae: 0.2748 - val_loss: 0.1048 - val_mae: 0.2380
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0873 - mae: 0.2147 - val_loss: 0.0924 - val_mae: 0.2207
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0787 - mae: 0.2017 - val_loss: 0.0865 - val_mae: 0.2095
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0742 - mae: 0.1941 - val_loss: 0.0838 - val_mae: 0.2059
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0716 - mae: 0.1898 - val_loss: 0.0826 - val_mae: 0.2063
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0701 - mae: 0.1872 - val_loss: 0.0825 - val_mae: 0.2064
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0688 - mae: 0.1852 - val_loss: 0.0826 - val_mae: 0.2032
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0682 - mae: 0.1843 - val_loss: 0.0816 - val_mae: 0.2041
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━

In [61]:
rnn32_pred_scaled = rnn_32.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rnn32_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21737, 24)


In [62]:
rnn32_pred = target_scaler.inverse_transform(
    rnn32_pred_scaled.reshape(-1, 1)
).reshape(rnn32_pred_scaled.shape)

rnn32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [63]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rnn32_mae = mean_absolute_error(
    rnn32_actual.flatten(),
    rnn32_pred.flatten()
)

rnn32_rmse = np.sqrt(
    mean_squared_error(
        rnn32_actual.flatten(),
        rnn32_pred.flatten()
    )
)

rnn32_mape = np.mean(
    np.abs(
        (rnn32_actual.flatten() - rnn32_pred.flatten())
        / rnn32_actual.flatten()
    )
) * 100

rnn32_r2 = r2_score(
    rnn32_actual.flatten(),
    rnn32_pred.flatten()
)

rnn32_bias = np.mean(
    rnn32_pred.flatten() -
    rnn32_actual.flatten()
)

print("=== RNN 32 NEURONS ===")
print("MAE :", rnn32_mae)
print("RMSE:", rnn32_rmse)
print("MAPE:", rnn32_mape)
print("R²  :", rnn32_r2)
print("Bias:", rnn32_bias)

=== RNN 32 NEURONS ===
MAE : 1410.3121300246148
RMSE: 1985.078525099613
MAPE: 4.47974784559913
R²  : 0.9050554097075454
Bias: 226.44287729357634


**batch size(32)**

In [64]:
rnn_batch32 = Sequential([
    SimpleRNN(64, input_shape=(24, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_batch32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch32 = rnn_batch32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=32,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 21s 6ms/step - loss: 0.1107 - mae: 0.2397 - val_loss: 0.0949 - val_mae: 0.2226
Epoch 2/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0751 - mae: 0.1964 - val_loss: 0.0830 - val_mae: 0.2052
Epoch 3/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 38s 6ms/step - loss: 0.0703 - mae: 0.1882 - val_loss: 0.0821 - val_mae: 0.2034
Epoch 4/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - loss: 0.0680 - mae: 0.1842 - val_loss: 0.0829 - val_mae: 0.2060
Epoch 5/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - loss: 0.0663 - mae: 0.1814 - val_loss: 0.0811 - val_mae: 0.2048
Epoch 6/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - loss: 0.0650 - mae: 0.1794 - val_loss: 0.0764 - val_mae: 0.1961
Epoch 7/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.0640 - mae: 0.1778 - val_loss: 0.0758 - val_mae: 0.1958
Epoch 8/15
3176/3176 ━━━━━━━━━━━━━━━━━━━━ 33s 10ms/step - loss: 0.0629 - mae: 0.1763 - val_loss: 0.0770 - val_mae: 0.1980
Epoch 9/15
3176/3176 ━━━━━━━

In [66]:
rnn_batch32_pred_scaled = rnn_batch32.predict(
    X_test_seq,
    batch_size=32
)

print("Prediction shape:", rnn_batch32_pred_scaled.shape)

680/680 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
Prediction shape: (21737, 24)


In [67]:
rnnbatch32_pred = target_scaler.inverse_transform(
    rnn_batch32_pred_scaled.reshape(-1, 1)
).reshape(rnn_batch32_pred_scaled.shape)

rnnbatch32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [69]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rnnb32_mae = mean_absolute_error(
    rnnbatch32_actual.flatten(),
    rnnbatch32_pred.flatten()
)

rnnb32_rmse = np.sqrt(
    mean_squared_error(
        rnnbatch32_actual.flatten(),
        rnnbatch32_pred.flatten()
    )
)

rnnb32_mape = np.mean(
    np.abs(
        (rnnbatch32_actual.flatten() - rnnbatch32_pred.flatten())
        / rnnbatch32_actual.flatten()
    )
) * 100

rnnb32_r2 = r2_score(
    rnnbatch32_actual.flatten(),
    rnnbatch32_pred.flatten()
)

rnnb32_bias = np.mean(
    rnnbatch32_pred.flatten() -
    rnnbatch32_actual.flatten()
)

print("=== RNN batch32 NEURONS ===")
print("MAE :", rnnb32_mae)
print("RMSE:", rnnb32_rmse)
print("MAPE:", rnnb32_mape)
print("R²  :", rnnb32_r2)
print("Bias:", rnnb32_bias)

=== RNN batch32 NEURONS ===
MAE : 1387.2739250699174
RMSE: 1961.7285185939083
MAPE: 4.396921367341186
R²  : 0.907275894192556
Bias: 200.6896267575759


**batchsize128**

In [70]:
rnn_batch128 = Sequential([
    SimpleRNN(64, input_shape=(24, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

rnn_batch128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch128 = rnn_batch128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=128,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.1541 - mae: 0.2797 - val_loss: 0.1023 - val_mae: 0.2361
Epoch 2/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.0831 - mae: 0.2092 - val_loss: 0.0907 - val_mae: 0.2171
Epoch 3/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0749 - mae: 0.1965 - val_loss: 0.0828 - val_mae: 0.2057
Epoch 4/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - loss: 0.0710 - mae: 0.1899 - val_loss: 0.0807 - val_mae: 0.2041
Epoch 5/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.0688 - mae: 0.1862 - val_loss: 0.0802 - val_mae: 0.1999
Epoch 6/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 0.0669 - mae: 0.1831 - val_loss: 0.0784 - val_mae: 0.1996
Epoch 7/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - loss: 0.0658 - mae: 0.1812 - val_loss: 0.0814 - val_mae: 0.2020
Epoch 8/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.0646 - mae: 0.1792 - val_loss: 0.0771 - val_mae: 0.1973
Epoch 9/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step 

In [71]:
rnn_batch128_pred_scaled = rnn_batch128.predict(
    X_test_seq,
    batch_size=128
)

print("Prediction shape:", rnn_batch128_pred_scaled.shape)

170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Prediction shape: (21737, 24)


In [72]:
rnnbatch128_pred = target_scaler.inverse_transform(
    rnn_batch128_pred_scaled.reshape(-1, 1)
).reshape(rnn_batch128_pred_scaled.shape)

rnnbatch128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [73]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rnnb128_mae = mean_absolute_error(
    rnnbatch128_actual.flatten(),
    rnnbatch128_pred.flatten()
)

rnnb128_rmse = np.sqrt(
    mean_squared_error(
        rnnbatch128_actual.flatten(),
        rnnbatch128_pred.flatten()
    )
)

rnnb128_mape = np.mean(
    np.abs(
        (rnnbatch128_actual.flatten() - rnnbatch128_pred.flatten())
        / rnnbatch128_actual.flatten()
    )
) * 100

rnnb128_r2 = r2_score(
    rnnbatch128_actual.flatten(),
    rnnbatch128_pred.flatten()
)

rnnb128_bias = np.mean(
    rnnbatch128_pred.flatten() -
    rnnbatch128_actual.flatten()
)

print("=== RNN batch128  ===")
print("MAE :", rnnb128_mae)
print("RMSE:", rnnb128_rmse)
print("MAPE:", rnnb128_mape)
print("R²  :", rnnb128_r2)
print("Bias:", rnnb128_bias)

=== RNN batch128  ===
MAE : 1399.362004928143
RMSE: 1976.6945657083681
MAPE: 4.432770661562494
R²  : 0.9058557111624999
Bias: 102.18250054016362


**HyperParameter Tuning**

In [77]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.6 MB/s eta 0:00:00


In [78]:
import keras_tuner as kt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [74]:
def build_rnn(hp):

    model = Sequential()

    model.add(
        SimpleRNN(
            units=hp.Choice(
                "rnn_units",
                values=[32, 64, 128]
            ),
            input_shape=(24, 15)
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )

    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [79]:
tuner = kt.RandomSearch(
    build_rnn,
    objective="val_loss",
    max_trials=6,
    executions_per_trial=1,
    directory="hyperparameter_tuning",
    project_name="rnn_24_to_24"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [80]:
tuner.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Trial 6 Complete [00h 02m 20s]
val_loss: 0.0905381590127945

Best val_loss So Far: 0.07468206435441971
Total elapsed time: 00h 14m 42s


In [81]:
best_hp = tuner.get_best_hyperparameters(
    num_trials=1
)[0]

print("BEST HYPERPARAMETERS")

print(
    "RNN Units:",
    best_hp.get("rnn_units")
)

print(
    "Dense Units:",
    best_hp.get("dense_units")
)

print(
    "Learning Rate:",
    best_hp.get("learning_rate")
)

BEST HYPERPARAMETERS
RNN Units: 64
Dense Units: 32
Learning Rate: 0.001


In [82]:
best_model = tuner.get_best_models(
    num_models=1
)[0]

best_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │           792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,992 (31.22 KB)

 Trainable params: 7,992 (31.22 KB)

 Non-trainable params: 0 (0.00 B)

In [83]:
history_best = best_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0611 - mae: 0.1743 - val_loss: 0.0774 - val_mae: 0.1973
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0605 - mae: 0.1734 - val_loss: 0.0771 - val_mae: 0.1992
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0602 - mae: 0.1729 - val_loss: 0.0768 - val_mae: 0.1978
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0600 - mae: 0.1725 - val_loss: 0.0793 - val_mae: 0.2007
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0597 - mae: 0.1721 - val_loss: 0.0755 - val_mae: 0.1944
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0595 - mae: 0.1719 - val_loss: 0.0770 - val_mae: 0.1975
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0592 - mae: 0.1713 - val_loss: 0.0742 - val_mae: 0.1921
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0589 - mae: 0.1709 - val_loss: 0.0746 - val_mae: 0.1943
Epoch 9/15
1588/1588 ━━━━━━━━━━━━━━━━━━

In [84]:
best_pred_scaled = best_model.predict(
    X_test_seq,
    batch_size=64
)

best_pred = target_scaler.inverse_transform(
    best_pred_scaled.reshape(-1, 1)
).reshape(best_pred_scaled.shape)

best_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [85]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

best_mae = mean_absolute_error(
    best_actual.flatten(),
    best_pred.flatten()
)

best_rmse = np.sqrt(
    mean_squared_error(
        best_actual.flatten(),
        best_pred.flatten()
    )
)

best_mape = np.mean(
    np.abs(
        (best_actual.flatten() - best_pred.flatten())
        / best_actual.flatten()
    )
) * 100

best_r2 = r2_score(
    best_actual.flatten(),
    best_pred.flatten()
)

best_bias = np.mean(
    best_pred.flatten() -
    best_actual.flatten()
)

print("HYPERPARAMETER TUNED RNN")
print("MAE :", best_mae)
print("RMSE:", best_rmse)
print("MAPE:", best_mape)
print("R²  :", best_r2)
print("Bias:", best_bias)

HYPERPARAMETER TUNED RNN
MAE : 1502.1588268103073
RMSE: 2068.5658742427486
MAPE: 4.816308814578437
R²  : 0.8969012134051946
Bias: 495.7678921709132


**additional lags**

In [119]:
# Additional lag features

PJME_data["Lag_2"] = PJME_data["PJME_MW"].shift(2)
PJME_data["Lag_3"] = PJME_data["PJME_MW"].shift(3)
PJME_data["Lag_6"] = PJME_data["PJME_MW"].shift(6)
PJME_data["Lag_12"] = PJME_data["PJME_MW"].shift(12)
PJME_data["Lag_72"] = PJME_data["PJME_MW"].shift(72)
PJME_data["Lag_336"] = PJME_data["PJME_MW"].shift(336)

In [120]:
PJME_data = PJME_data.dropna().copy()

print("Missing values:")
print(PJME_data.isnull().sum().sum())

print("Shape:")
print(PJME_data.shape)

Missing values:
0
Shape:
(144552, 21)


In [121]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",

    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_24",
    "Lag_48",
    "Lag_72",
    "Lag_168",
    "Lag_336",

    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

print("Number of features:", len(features))

Number of features: 21


In [122]:
n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = PJME_data.iloc[:train_end]
val_data = PJME_data.iloc[train_end:val_end]
test_data = PJME_data.iloc[val_end:]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (101186, 21)
Validation: (21683, 21)
Test: (21683, 21)


In [123]:
X_train = train_data[features]
y_train = train_data[["PJME_MW"]]

X_val = val_data[features]
y_val = val_data[["PJME_MW"]]

X_test = test_data[features]
y_test = test_data[["PJME_MW"]]

In [124]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train)
y_val_scaled = target_scaler.transform(y_val)
y_test_scaled = target_scaler.transform(y_test)

print("Scaling completed.")

Scaling completed.


In [125]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=24,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=24,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=24,
    output_steps=24
)

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101139, 24, 21)
y_train: (101139, 24)
X_val: (21636, 24, 21)
y_val: (21636, 24)
X_test: (21636, 24, 21)
y_test: (21636, 24)


In [126]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

rnn_lags = Sequential([
    SimpleRNN(
        64,
        input_shape=(24, 21)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(24)
])

rnn_lags.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

rnn_lags.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_6 (SimpleRNN)        │ (None, 64)             │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,224 (43.84 KB)

 Trainable params: 11,224 (43.84 KB)

 Non-trainable params: 0 (0.00 B)

In [127]:
history_lags = rnn_lags.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1581/1581 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - loss: 0.1216 - mae: 0.2492 - val_loss: 0.0908 - val_mae: 0.2190
Epoch 2/15
1581/1581 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0773 - mae: 0.1992 - val_loss: 0.0888 - val_mae: 0.2149
Epoch 3/15
1581/1581 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0707 - mae: 0.1888 - val_loss: 0.0825 - val_mae: 0.2072
Epoch 4/15
1581/1581 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - loss: 0.0675 - mae: 0.1833 - val_loss: 0.0777 - val_mae: 0.1986
Epoch 5/15
1581/1581 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0659 - mae: 0.1808 - val_loss: 0.0757 - val_mae: 0.1974
Epoch 6/15
1581/1581 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0646 - mae: 0.1786 - val_loss: 0.0755 - val_mae: 0.1967
Epoch 7/15
1581/1581 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 0.0625 - mae: 0.1753 - val_loss: 0.0785 - val_mae: 0.2028
Epoch 8/15
1581/1581 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - loss: 0.0616 - mae: 0.1740 - val_loss: 0.0773 - val_mae: 0.2008
Epoch 9/15
1581/1581 ━━━━━━━━━━━━━━━━━━

In [128]:
pred_scaled = rnn_lags.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", pred_scaled.shape)

339/339 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21636, 24)


In [129]:
pred = target_scaler.inverse_transform(
    pred_scaled.reshape(-1, 1)
).reshape(pred_scaled.shape)

actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [130]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

mae = mean_absolute_error(
    actual.flatten(),
    pred.flatten()
)

rmse = np.sqrt(
    mean_squared_error(
        actual.flatten(),
        pred.flatten()
    )
)

mape = np.mean(
    np.abs(
        (actual.flatten() - pred.flatten())
        / actual.flatten()
    )
) * 100

r2 = r2_score(
    actual.flatten(),
    pred.flatten()
)

bias = np.mean(
    pred.flatten() - actual.flatten()
)

print("========== RNN + ADDITIONAL LAGS ==========")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

========== RNN + ADDITIONAL LAGS ==========
MAE : 1518.6297064285338
RMSE: 2115.73965431117
MAPE: 4.899385060780936
R²  : 0.8921897490234234
Bias: 368.47112271678037


In [156]:
import pandas as pd

phase6_results = [

    # Early Stopping
    {
        "Technique": "Early Stopping",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1390.3678448612964,
    "RMSE": 1949.1159618860881,
    "MAPE": 4.410774413858794,
    "R2": 0.9084643650051681,
    "Bias": 120.65475509308246
    },

    # Dropout
    {
        "Technique": "Dropout",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1471.529424933611,
        "RMSE": 2009.527278024123,
        "MAPE":4.691583628341732,
        "R2": 0.9027022820804059,
        "Bias": 117.89846316413617
    },



    # Batch Normalization
    {
        "Technique": "Batch Normalization",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1388.6084414431793,
        "RMSE": 1960.869031599301,
        "MAPE": 4.408847433340167,
        "R2": 0.9073571263344717,
        "Bias": 216.50879045895846
    },

    # Adam
    {
        "Technique": "Adam",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1412.8646858513553,
        "RMSE": 1960.2076349436081,
        "MAPE": 4.477067885744956,
        "R2": 0.907419612254838,
        "Bias": 138.76057845742451
    },

    # RMSprop
    {
        "Technique": "RMSprop",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1387.1081347502004,
        "RMSE": 1931.1706843661027,
        "MAPE": 4.41234224260603,
        "R2": 0.9101421211213101,
        "Bias": 273.0465344812776
    },

    # RMSprop + Learning Rate
    {
        "Technique": "RMSprop + Learning Rate Scheduler",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1380.455857086755,
        "RMSE": 1947.0097018210615,
        "MAPE": 4.373047915495958,
        "R2": 0.9086620891892203,
        "Bias": 183.9371604022663
    },

    # SGD
    {
        "Technique": "SGD",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1611.0813879169339,
        "RMSE": 2180.9567945231242,
        "MAPE": 5.1341236813930715,
        "R2": 0.8853935746511137,
        "Bias": 159.78651111047816
    },

    # Deep RNN
    {
        "Technique": "Deep RNN",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1499.4187534323676,
        "RMSE": 2093.1793458589927,
        "MAPE": 4.784466855917797,
        "R2": 0.8944331108299858,
        "Bias": 383.87631982172536
    },

    # 128 Neurons
    {
        "Technique": "128 Neurons",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1458.0408453731266,
        "RMSE": 2046.0232695734032,
        "MAPE": 4.657677636162435,
        "R2": 0.8991360481426458,
        "Bias": 522.2727100125278
    },

    # 32 Neurons
    {
        "Technique": "32 Neurons",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1410.3121300246148,
        "RMSE": 1985.078525099613,
        "MAPE": 4.47974784559913,
        "R2": 0.9050554097075454,
        "Bias": 226.44287729357634
    },

    # Batch Size 32
    {
        "Technique": "Batch Size 32",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1387.2739250699174,
        "RMSE": 1961.7285185939083,
        "MAPE": 4.396921367341186,
        "R2": 0.907275894192556,
        "Bias": 200.6896267575759
    },

    # Batch Size 128
    {
        "Technique": "Batch Size 128",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1399.362004928143,
        "RMSE": 1976.6945657083681,
        "MAPE": 4.432770661562494,
        "R2": 0.9058557111624999,
        "Bias": 102.18250054016362
    },

    # Hyperparameter Tuning
    {
        "Technique": "Hyperparameter Tuning",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1502.1588268103073,
        "RMSE": 2068.5658742427486,
        "MAPE": 4.816308814578437,
        "R2": 0.8969012134051946,
        "Bias": 495.7678921709132
    },


    # Additional Lags
    {
        "Technique": "Additional Lags",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1518.6297064285338,
        "RMSE": 2115.73965431117,
        "MAPE": 4.899385060780936,
        "R2": 0.8921897490234234,
        "Bias": 368.47112271678037
    },

    # Best Baseline
    {
        "Technique": "Baseline",
        "Model": "RNN",
        "Input_Output": "24 → 24",
        "MAE": 1376.7156520411265,
        "RMSE": 1926.8904657690284,
        "MAPE": 4.347443014548393,
        "R2": 0.9105399990964322,
        "Bias": 78.35691115307425
    }
]

phase6_df = pd.DataFrame(phase6_results)

# Sort by MAPE
phase6_df = phase6_df.sort_values(
    by="MAPE",
    ascending=True
).reset_index(drop=True)

phase6_df.insert(
    0,
    "Rank",
    range(1, len(phase6_df) + 1)
)

# Display
display(phase6_df)



,Rank,Technique,Model,Input_Output,MAE,RMSE,MAPE,R2,Bias
0,1,Baseline,RNN,24 → 24,1376.715652,1926.890466,4.347443,0.910540,78.356911
1,2,RMSprop + Learning Rate Scheduler,RNN,24 → 24,1380.455857,1947.009702,4.373048,0.908662,183.937160
2,3,Batch Size 32,RNN,24 → 24,1387.273925,1961.728519,4.396921,0.907276,200.689627
3,4,Batch Normalization,RNN,24 → 24,1388.608441,1960.869032,4.408847,0.907357,216.508790
4,5,Early Stopping,RNN,24 → 24,1390.367845,1949.115962,4.410774,0.908464,120.654755
5,6,RMSprop,RNN,24 → 24,1387.108135,1931.170684,4.412342,0.910142,273.046534
6,7,Batch Size 128,RNN,24 → 24,1399.362005,1976.694566,4.432771,0.905856,102.182501
7,8,Adam,RNN,24 → 24,1412.864686,1960.207635,4.477068,0.907420,138.760578
8,9,32 Neurons,RNN,24 → 24,1410.312130,1985.078525,4.479748,0.905055,226.442877
9,10,128 Neurons,RNN,24 → 24,1458.040845,2046.023270,4.657678,0.899136,522.272710


In [157]:
phase6_df.to_csv(
    "processed_data/Phase6_All_Experiments.csv",
    index=False
)

print("Phase 6 CSV saved successfully.")
print("File: processed_data/Phase6_All_Experiments.csv")

Phase 6 CSV saved successfully.
File: processed_data/Phase6_All_Experiments.csv
